# 02 — Clean (step 1: resolve CLAUDE.md -> AGENTS.md)

The unit is the **repo**. Each repo gets one `content_resolved` and a `resolved_from` label. **`CLAUDE.md` takes priority; if the repo has no `CLAUDE.md`, use its `AGENTS.md`.**
- `CLAUDE.md` exists and is substantive -> use it.
- `CLAUDE.md` is a pure pointer (symlink to AGENTS.md, or short and mentions AGENTS.md) -> use `AGENTS.md`.
- `CLAUDE.md` imports `@AGENTS.md` but has its own content -> `CLAUDE.md` text + `AGENTS.md` text, labelled `CLAUDE.md+AGENTS.md`.
- No `CLAUDE.md`, only `AGENTS.md` -> use `AGENTS.md`.
- Pointer but the target text is missing -> `unresolved` (kept, raw text unchanged).

One hop only; other `@imports` are not followed. `content_raw` is never overwritten. Dedup and secret stripping come in later steps of this notebook.

In [1]:
import re
from pathlib import Path
import pandas as pd

IN_PATH = Path("data/raw/pilot_50.parquet")
OUT_PATH = Path("data/interim/resolved.parquet")
POINTER_MAX_CHARS = 500          # a CLAUDE.md shorter than this that mentions AGENTS.md is treated as a pointer
IMPORT_RE = re.compile(r"(^|\s)@\.?/?AGENTS\.md\b", re.I)
MENTION_RE = re.compile(r"AGENTS\.md", re.I)

raw = pd.read_parquet(IN_PATH)
print(raw.shape)

(50, 25)


## 1. One row per repo

In [2]:
repos = raw.drop_duplicates("full_name").copy()    # one row per repo; claude_text / agents_text come straight from the scrape
repos["claude_symlink_to_agents"] = repos["claude_symlink_target"].fillna("").str.contains("AGENTS.md", case=False)
print(len(repos), "repos")

50 repos


## 2. Detect pointers and resolve

In [3]:
def resolve(r):
    c = None if pd.isna(r.claude_text) else r.claude_text
    a = None if pd.isna(r.agents_text) else r.agents_text
    if c is None and not r.claude_symlink_to_agents:
        return pd.Series({"is_redirect": False, "resolved_from": "AGENTS.md", "content_resolved": a})
    body = (c or "").strip()
    is_import = bool(IMPORT_RE.search(body))
    pure_pointer = r.claude_symlink_to_agents or (len(body) < POINTER_MAX_CHARS and bool(MENTION_RE.search(body)))
    if not (pure_pointer or is_import):
        return pd.Series({"is_redirect": False, "resolved_from": "CLAUDE.md", "content_resolved": c})
    if a is None:
        return pd.Series({"is_redirect": True, "resolved_from": "unresolved", "content_resolved": c})
    if pure_pointer:
        return pd.Series({"is_redirect": True, "resolved_from": "AGENTS.md", "content_resolved": a})
    return pd.Series({"is_redirect": True, "resolved_from": "CLAUDE.md+AGENTS.md", "content_resolved": c + "\n\n" + a})

repos[["is_redirect", "resolved_from", "content_resolved"]] = repos.apply(resolve, axis=1)
repos["n_chars_resolved"] = repos["content_resolved"].str.len()
repos["no_claude_md"] = ~repos["has_claude_md"]
print(repos["resolved_from"].value_counts())
print("redirects:", repos["is_redirect"].sum(), "| AGENTS.md-only repos:", repos["no_claude_md"].sum())

resolved_from
CLAUDE.md              32
AGENTS.md              16
CLAUDE.md+AGENTS.md     2
Name: count, dtype: int64
redirects: 6 | AGENTS.md-only repos: 12


## 3. Eyeball the redirects and the short files

In [4]:
repos.loc[repos.is_redirect | (repos.n_chars_resolved < 400),
          ["full_name", "primary_file", "has_both", "is_redirect", "resolved_from", "n_chars", "n_chars_resolved"]]

,full_name,primary_file,has_both,is_redirect,resolved_from,n_chars,n_chars_resolved
3,NachoSoto/AsyncImageView,AGENTS.md,False,False,AGENTS.md,202,202
5,huang-sh/PiX,AGENTS.md,False,False,AGENTS.md,353,353
15,ruby-no-kai/regional.rubykaigi.org,CLAUDE.md,True,True,AGENTS.md,11,1171
26,cdavid817/vanehub-ai,CLAUDE.md,True,True,CLAUDE.md+AGENTS.md,609,7643
29,sveneisenschmidt/n8n-openai-bridge,CLAUDE.md,True,True,AGENTS.md,11,8145
32,darksworm/argonaut,CLAUDE.md,False,False,CLAUDE.md,265,265
34,adrianiftode/Assertions.Web,CLAUDE.md,False,False,CLAUDE.md,140,140
37,rmordechay/glsl-plugin-idea,CLAUDE.md,True,False,CLAUDE.md,59,59
39,cisagov/ScubaConnect,CLAUDE.md,True,True,AGENTS.md,10,2586
44,delphix/delphix-platform,CLAUDE.md,False,False,CLAUDE.md,275,275


## 4. Checkpoint

In [5]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
repos.drop(columns=["claude_text", "agents_text"]).to_parquet(OUT_PATH, index=False)
print("saved", OUT_PATH, repos.shape)

saved data/interim/resolved.parquet (50, 31)


## 5. Strip secrets
Regex-based redaction of well-known key formats plus `key = "value"` style assignments. Matches are replaced with `[REDACTED:<type>]`. Values are never printed. Placeholders such as `your-api-key-here` are left alone. This is a best-effort scan, not a guarantee; the dataset card says so.

In [6]:
SECRET_PATTERNS = [
    ("private_key", re.compile(r"-----BEGIN [A-Z ]*PRIVATE KEY-----[\s\S]*?-----END [A-Z ]*PRIVATE KEY-----")),
    ("github_token", re.compile(r"\b(?:ghp|gho|ghu|ghs|ghr)_[A-Za-z0-9]{36,}\b|\bgithub_pat_[A-Za-z0-9_]{50,}\b")),
    ("aws_access_key", re.compile(r"\b(?:AKIA|ASIA)[0-9A-Z]{16}\b")),
    ("anthropic_key", re.compile(r"\bsk-ant-[A-Za-z0-9_\-]{20,}\b")),
    ("openai_key", re.compile(r"\bsk-(?:proj-)?[A-Za-z0-9_\-]{32,}\b")),
    ("slack_token", re.compile(r"\bxox[baprs]-[A-Za-z0-9\-]{10,}\b")),
    ("google_api_key", re.compile(r"\bAIza[0-9A-Za-z_\-]{35}\b")),
    ("stripe_key", re.compile(r"\b[sr]k_live_[0-9A-Za-z]{20,}\b")),
    ("jwt", re.compile(r"\beyJ[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,}\b")),
]
ASSIGN_RE = re.compile(r"(?i)\b(api[_-]?key|secret|token|passwd|password)\b(\s*[:=]\s*['\"])([A-Za-z0-9/+_\-]{20,})(['\"])")
PLACEHOLDER_WORDS = ("example", "your", "xxx", "placeholder", "changeme", "replace", "dummy", "sample", "<", "...")

def redact(text):
    found = []
    for kind, rx in SECRET_PATTERNS:
        text, n = rx.subn(f"[REDACTED:{kind}]", text)
        found += [kind] * n
    def _assign(m):
        if any(w in m.group(3).lower() for w in PLACEHOLDER_WORDS):
            return m.group(0)
        found.append("assigned_secret")
        return f"{m.group(1)}{m.group(2)}[REDACTED:assigned_secret]{m.group(4)}"
    text = ASSIGN_RE.sub(_assign, text)
    return text, found

out = repos["content_resolved"].map(redact)
repos["content_clean"] = out.map(lambda t: t[0])
repos["secret_types"] = out.map(lambda t: sorted(set(t[1])))
repos["n_secrets"] = out.map(lambda t: len(t[1]))
print("files with a redacted secret:", (repos.n_secrets > 0).sum(), "| total redactions:", repos.n_secrets.sum())
print(repos.loc[repos.n_secrets > 0, ["full_name", "n_secrets", "secret_types"]].to_string())

files with a redacted secret: 0 | total redactions: 0
Empty DataFrame
Columns: [full_name, n_secrets, secret_types]
Index: []


## 6. Basic filters (flag first, drop only empty/tiny)
- **Empty/tiny:** fewer than `MIN_CHARS` characters after stripping whitespace -> dropped from the clean set and logged.
- **Non-English:** flagged (`non_english_likely`: over 20% of letters are non-ASCII), not dropped. The dataset card says the corpus is English-dominant; whether to exclude them is a question for Kaiser.

In [7]:
MIN_CHARS = 100
repos["n_chars_clean"] = repos["content_clean"].fillna("").str.strip().str.len()
letters = repos["content_clean"].fillna("").map(lambda t: [ch for ch in t if ch.isalpha()])
repos["non_english_likely"] = letters.map(lambda L: bool(L) and sum(not c.isascii() for c in L) / len(L) > 0.20)
repos["is_tiny"] = repos["n_chars_clean"] < MIN_CHARS
print("tiny (<%d chars):" % MIN_CHARS, repos.is_tiny.sum(), "| likely non-English:", repos.non_english_likely.sum())
repos.loc[repos.is_tiny | repos.non_english_likely,
          ["full_name", "primary_file", "resolved_from", "n_chars_clean", "is_tiny", "non_english_likely"]]

tiny (<100 chars): 1 | likely non-English: 1


,full_name,primary_file,resolved_from,n_chars_clean,is_tiny,non_english_likely
26,cdavid817/vanehub-ai,CLAUDE.md,CLAUDE.md+AGENTS.md,7642,False,True
37,rmordechay/glsl-plugin-idea,CLAUDE.md,CLAUDE.md,59,True,False


## 7. Dedup
Texts are lower-cased and whitespace-collapsed. **Exact** duplicates share a hash. **Near** duplicates have word-5-gram Jaccard similarity of at least `NEAR_DUP_JACCARD`. Within a duplicate group the most-starred repo is kept; the others are dropped and logged with the repo they duplicate.

In [8]:
import hashlib
NEAR_DUP_JACCARD = 0.85

def norm(t): return re.sub(r"\s+", " ", (t or "").lower()).strip()
def shingles(t, k=5):
    w = norm(t).split()
    return {" ".join(w[i:i + k]) for i in range(max(len(w) - k + 1, 1))}

cand = repos[~repos.is_tiny].sort_values("stars", ascending=False)
kept, dup_of, dup_kind = [], {}, {}
sh, hashes = {}, {}
for idx, r in cand.iterrows():
    h = hashlib.sha1(norm(r.content_clean).encode()).hexdigest()
    s = shingles(r.content_clean)
    if h in hashes:
        dup_of[idx], dup_kind[idx] = hashes[h], "exact"
        continue
    match = next((k for k in kept if len(s & sh[k]) / max(len(s | sh[k]), 1) >= NEAR_DUP_JACCARD), None)
    if match is not None:
        dup_of[idx], dup_kind[idx] = match, "near"
        continue
    kept.append(idx); sh[idx] = s; hashes[h] = idx

repos["duplicate_of"] = pd.Series({i: repos.loc[j, "full_name"] for i, j in dup_of.items()})
repos["duplicate_kind"] = pd.Series(dup_kind)
print("exact:", sum(v == "exact" for v in dup_kind.values()), "| near:", sum(v == "near" for v in dup_kind.values()))
repos.loc[repos.duplicate_of.notna(), ["full_name", "duplicate_of", "duplicate_kind", "n_chars_clean"]]

exact: 0 | near: 0


,full_name,duplicate_of,duplicate_kind,n_chars_clean


## 8. Clean set + drop log + checkpoint

In [9]:
repos["drop_reason"] = None
repos.loc[repos.is_tiny, "drop_reason"] = "tiny"
repos.loc[repos.duplicate_of.notna(), "drop_reason"] = "duplicate_" + repos["duplicate_kind"].fillna("")
RAW_TEXT_COLS = ["content_raw", "claude_text", "agents_text", "content_resolved"]   # may contain secrets: never leave this notebook
clean = repos[repos.drop_reason.isna()].drop(columns=RAW_TEXT_COLS).reset_index(drop=True)
drop_log = repos[repos.drop_reason.notna()][["full_name", "primary_file", "drop_reason", "duplicate_of", "n_chars_clean"]]
print(len(repos), "->", len(clean), "repos kept")
print(drop_log.to_string() if len(drop_log) else "nothing dropped")
clean.to_parquet("data/interim/clean.parquet", index=False)
drop_log.to_parquet("data/interim/drop_log.parquet", index=False)

50 -> 49 repos kept
                      full_name primary_file drop_reason duplicate_of  n_chars_clean
37  rmordechay/glsl-plugin-idea    CLAUDE.md        tiny          NaN             59


## 9. Descriptive numbers for the pilot (for the Kaiser check-in)

In [10]:
clean["n_words"] = clean["content_clean"].str.split().str.len()
print("repos:", len(clean))
print("\ngroup (primary_file):"); print(clean.primary_file.value_counts().to_string())
print("\nresolved_from:"); print(clean.resolved_from.value_counts().to_string())
print(f"\nredirect share: {clean.is_redirect.mean():.0%} | both files present: {clean.has_both.mean():.0%} | likely non-English: {clean.non_english_likely.mean():.0%}")
print(f"has_tests: {clean.has_tests.mean():.0%} | has_ci: {clean.has_ci.mean():.0%}")
print("\nlength (chars) and words:")
print(clean[["n_chars_clean", "n_words"]].describe(percentiles=[.25, .5, .75, .9]).round(0).to_string())
print("\nlength by group (median chars):"); print(clean.groupby("primary_file").n_chars_clean.median().to_string())
print("\ntop languages:"); print(clean.language.value_counts().head(8).to_string())
print("\nstars:"); print(clean.stars.describe(percentiles=[.25, .5, .75]).round(0).to_string())
print("\nstar bins:"); print(clean.star_bin.value_counts().sort_index().to_string())

repos: 49

group (primary_file):


primary_file
CLAUDE.md    37
AGENTS.md    12

resolved_from:
resolved_from
CLAUDE.md              31
AGENTS.md              16
CLAUDE.md+AGENTS.md     2

redirect share: 12% | both files present: 24% | likely non-English: 2%
has_tests: 57% | has_ci: 84%

length (chars) and words:
       n_chars_clean  n_words
count           49.0     49.0
mean         11918.0   1602.0
std          25141.0   3610.0
min            139.0     22.0
25%           1198.0    179.0
50%           3854.0    523.0
75%          12612.0   1470.0
90%          26218.0   3486.0
max         165009.0  24016.0

length by group (median chars):
primary_file
AGENTS.md    2795.5
CLAUDE.md    5619.0

top languages:
language
TypeScript    9
Python        7
Rust          6
Go            5
C#            4
JavaScript    3
Ruby          2
Swift         2

stars:
count     49.0
mean      94.0
std       91.0
min        2.0
25%       36.0
50%       65.0
75%      129.0
max      473.0

star bins:
star_bin
0-10         6
10-100      25
1